## NBA 3.1 | Patient Features EDA (RX / DX / PX / LAB)

QA of every output written by the four job tasks `RX_Patient_Features`, `DX_Patient_Features`, `PX_Patient_Features`, `LAB_Patient_Features`
(`.../kerendia_nba_3_1_volume/notebook_output/patient_features/{rx,dx,px,lab}/*_features`).

| # | Check | What is tested |
| --- | --- | --- |
| 1 | Fill rates | % non-null and % non-zero for every feature column |
| 2 | Ratio sanity | every percentage / ratio feature (`PERC_*`, `PCT_*`, `*_RATIO_*`, ...) is within [0, 1]; NaN / Inf scan |
| 3 | Cohort coverage | one row per `HCP_COHORT_ID`, no duplicates, all 28 cohorts (c1-c28) present; monthly tables: one row per HCP x cohort x month, 12 months per cohort |
| 4 | Universe coverage | distinct `PROVIDER_ID` vs the 314,920-HCP universe; which HCPs are missing (written to Delta) and HCPs outside the universe |
| 5 | Cross-notebook | which `HCP_COHORT_ID`s appear in RX / DX / PX / LAB and in which feature type, plus the subset rules they should follow |
| 6 | Range checks | count / frequency features above the 99.9th percentile, implausible maxima, negative counts, day-features outside the cohort window, date columns outside the cohort range |
| 7 | Schema drift | actual columns vs the columns `patient_features_skills/SKILL.md` says each output should contain (incl. NBA 3.1 removal notes) |

**Grain note.** First Occurrence and Recency are written at `HCP_COHORT_ID` grain. Frequency and Hospitalization are written at `PROVIDER_ID x COHORT x SVC_DT_M` (12 monthly rows per HCP-cohort). For checks 1, 2 (counts), 3, 4, 5 and 6 the monthly tables are reduced to the **model snapshot row** (`SVC_DT_M = lookback_end_date` of the cohort), i.e. the row that joins to `HCP_COHORT_ID` in the base table (SKILL.md Sec 8.2). Snapshots are materialised once under `_eda/snapshots/`.

All result tables are written to `.../notebook_output/patient_features/_eda/` and a scorecard is printed in the last cell.

In [0]:
%run "/Workspace/Users/wenjie.chen@bayer.com/kerendia_nba_3_0_develop/NBA 3.0 Ops - Pipeline/00_Helper_Notebooks/00_connection"

In [0]:
%run "/Workspace/Users/wenjie.chen@bayer.com/kerendia_nba_3_1_develop_sa/NBA 3.1 - Dev/02_Input_Data_Creation/00_event_list_config"

In [0]:
%run "/Workspace/Users/wenjie.chen@bayer.com/kerendia_nba_3_1_develop_sa/NBA 3.1 - Dev/00_Helper_Notebooks/00_cohort_generation"

In [0]:
import re, builtins          # builtins: the 00_connection helper overrides Python's sum
import pandas as pd
from functools import reduce
from pyspark.sql import functions as F

VOL               = "/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_1_volume"
FEATURE_BASE      = f"{VOL}/notebook_output/patient_features"
HCP_UNIVERSE_PATH = f"{VOL}/HCP_Universe"
EDA_OUT           = f"{FEATURE_BASE}/_eda"      # every EDA result table is written here (Delta)

EXPECTED_UNIVERSE = 314_920
EXPECTED_COHORTS  = [f"c{i}" for i in range(1, 29)]
DATA_START_DATE   = "2021-01-01"   # earliest claim in the full-history inputs is 2021-01-02
OUTLIER_FACTOR    = 10             # 'implausible' = max > OUTLIER_FACTOR x max(p99.9, 1)
CHUNK             = 300            # feature columns per aggregation pass

SOURCES = ["rx", "dx", "px", "lab"]
FEATURE_TYPES = {
    "rx":  ["first_occurrence", "recency", "frequency"],
    "dx":  ["first_occurrence", "recency", "frequency", "hospitalization"],
    "px":  ["first_occurrence", "recency", "frequency", "hospitalization"],
    "lab": ["first_occurrence", "recency", "frequency"],
}
GRAIN = {"first_occurrence": "hcp_cohort", "recency": "hcp_cohort",
         "frequency": "hcp_month", "hospitalization": "hcp_month"}
KEY_COLS = {"PROVIDER_ID", "COHORT", "HCP_COHORT_ID", "SVC_DT_M", "LOOKBACK_END_DATE"}

# ---- Expected schema, from patient_features_skills/SKILL.md (Sec 3-6, 8.3) incl. the NBA 3.1 removal notes ----
RECENCY_WINDOWS  = [30, 60, 90, 180, 270, 360]   # Sec 4
LAG_WINDOWS      = [1, 2, 3, 6, 12]              # Sec 5B
LOGRATIO_WINDOWS = [3, 6]                        # Sec 5C 3.1 note: LOGRATIO_FREQ_* for N=3,6 only, CHANGE_IN_FREQ_* dropped
PATIENT_WINDOWS  = [90, 180, 360]                # Sec 5 3.1 note: unique patients only for 90/180/360 d
HOSP_LAG_WINDOWS = [1, 6, 12]                    # Sec 6B
HOSP_PAT_WINDOWS = [30, 180, 360]                # Sec 6D
RECENCY_SUFFIX   = {"rx": "RX", "dx": "DX", "px": "PX", "lab": "LAB_DATA"}
PATIENT_PREFIX   = {"rx":  "NUM_OF_PATIENTS_THE_HCP_PRESCRIBED_WITH",
                    "dx":  "NUM_OF_PATIENTS_THE_HCP_DIAGNOSED_WITH",
                    "px":  "NUM_OF_PATIENTS_THE_HCP_DID_PROCEDURE",
                    "lab": "NUM_OF_PATIENTS_THE_HCP_DID_LAB_TEST"}
HOSP_EVENT       = {"dx": "ALL_DIAGNOSIS", "px": "ALL_PROCEDURES"}
HOSP_PAT_PREFIX  = {"dx": "NUM_OF_HOSPITALIZED_PATIENTS_THE_HCP_DIAGNOSED",
                    "px": "NUM_OF_HOSPITALIZED_PATIENTS_THE_HCP_DID_PROCEDURE"}

print(f"FEATURE_BASE : {FEATURE_BASE}")
print(f"EDA_OUT      : {EDA_OUT}")

In [0]:
NUMERIC_TYPES = ("int", "bigint", "double", "float", "smallint", "tinyint", "decimal", "long")

def eda_sanitize(name):
    """Same rule as sanitize_column_name in the four feature notebooks."""
    name = re.sub(r'[^A-Za-z0-9_]', '_', name.replace("+", "AND").replace("/", "_OR_"))
    return re.sub(r'_+', '_', name).strip('_').upper()

def num_cols(df):
    return [c for c, t in df.dtypes if c not in KEY_COLS and t.startswith(NUMERIC_TYPES)]

def chunks(lst, n):
    return [lst[i:i + n] for i in range(0, len(lst), n)]

def lbl(k):
    return f"{k[0]}_{k[1]}"

SCORECARD = []
def score(check, output, status, detail):
    SCORECARD.append({"CHECK": check, "OUTPUT": output, "STATUS": status, "DETAIL": str(detail)})

def save_pdf(pdf, name):
    if pdf is not None and len(pdf):
        (spark.createDataFrame(pdf.astype({c: str for c in pdf.columns if pdf[c].dtype == object}))
              .write.format("delta").mode("overwrite").option("overwriteSchema", "true")
              .save(f"{EDA_OUT}/{name}"))

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 250)

In [0]:
# ---- cohorts (claims family, offset 0) ----
cohorts = CohortGenerator("claims").load()
cohort_pd = pd.DataFrame({
    "COHORT":   cohorts["cohort_id"].astype(str).values,
    "LB_START": pd.to_datetime(cohorts["lookback_start_date"]).dt.date.values,
    "LB_END":   pd.to_datetime(cohorts["lookback_end_date"]).dt.date.values,
})
cohort_pd["N_MONTHS"] = [(e.year - s.year) * 12 + (e.month - s.month) + 1
                         for s, e in zip(cohort_pd.LB_START, cohort_pd.LB_END)]
cohort_sdf = spark.createDataFrame(cohort_pd)
print(f"Cohorts: {len(cohort_pd)} ({cohort_pd.COHORT.iloc[0]} {cohort_pd.LB_START.iloc[0]}->{cohort_pd.LB_END.iloc[0]} ... "
      f"{cohort_pd.COHORT.iloc[-1]} {cohort_pd.LB_START.iloc[-1]}->{cohort_pd.LB_END.iloc[-1]})")
if list(cohort_pd.COHORT) != EXPECTED_COHORTS:
    print("WARNING: cohort config does not hold exactly c1..c28")

# ---- HCP universe ----
universe  = spark.read.parquet(HCP_UNIVERSE_PATH)
flag_cols = [c for c in universe.columns if c.endswith("_FLAG")]
uni       = universe.groupBy("PROVIDER_ID").agg(*[F.max(c).alias(c) for c in flag_cols])
n_universe = uni.count()
print(f"HCP universe: {universe.count():,} rows, {n_universe:,} distinct PROVIDER_ID (expected {EXPECTED_UNIVERSE:,})")
score("4 Universe", "HCP_Universe", "PASS" if n_universe == EXPECTED_UNIVERSE else "FAIL",
      f"{n_universe:,} distinct HCPs vs expected {EXPECTED_UNIVERSE:,}")

# ---- feature outputs ----
outputs, missing_outputs = {}, []
for src, fts in FEATURE_TYPES.items():
    for ft in fts:
        path = f"{FEATURE_BASE}/{src}/{src}_{ft}_features"
        try:
            df = spark.read.format("delta").load(path)
            _ = df.columns                      # forces schema resolution
            outputs[(src, ft)] = df
        except Exception as e:
            missing_outputs.append((src, ft))
            print(f"MISSING  {src}/{src}_{ft}_features  -> {str(e)[:120]}")
            score("0 Load", f"{src}_{ft}", "FAIL", "output not readable / not written")
print(f"Loaded {len(outputs)} of {builtins.sum(len(v) for v in FEATURE_TYPES.values())} outputs")

# ---- model snapshot for monthly-grain tables (SVC_DT_M = cohort lookback_end_date) ----
snap_key = F.broadcast(cohort_sdf.select("COHORT", F.col("LB_END").alias("SVC_DT_M")))
analysis = {}
for k, df in outputs.items():
    if GRAIN[k[1]] == "hcp_cohort":
        analysis[k] = df
    else:
        snap_path = f"{EDA_OUT}/snapshots/{lbl(k)}"
        (df.join(snap_key, ["COHORT", "SVC_DT_M"], "inner")
           .withColumn("HCP_COHORT_ID", F.concat_ws("_", "PROVIDER_ID", "COHORT"))
           .write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(snap_path))
        analysis[k] = spark.read.format("delta").load(snap_path)

load_pdf = pd.DataFrame([{"SOURCE": k[0], "OUTPUT": k[1], "GRAIN": GRAIN[k[1]],
                          "N_COLUMNS": len(outputs[k].columns),
                          "N_FEATURES": len(num_cols(outputs[k])),
                          "ROWS_WRITTEN": outputs[k].count(),
                          "ROWS_ANALYSED": analysis[k].count()} for k in outputs])
print(load_pdf.to_string(index=False))

In [0]:
fill_records = []
for k, df in analysis.items():
    n = df.count()
    for ch in chunks(num_cols(df), CHUNK):
        exprs = []
        for i, c in enumerate(ch):
            exprs += [F.count(F.col(c)).alias(f"nn{i}"),
                      F.sum(F.when(F.col(c) != 0, 1).otherwise(0)).alias(f"nz{i}")]
        r = df.agg(*exprs).first()
        for i, c in enumerate(ch):
            nn, nz = (r[f"nn{i}"] or 0), (r[f"nz{i}"] or 0)
            fill_records.append({"SOURCE": k[0], "OUTPUT": k[1], "COLUMN": c, "ROWS": n,
                                 "PCT_NON_NULL": round(100 * nn / n, 4) if n else None,
                                 "PCT_NON_ZERO": round(100 * nz / n, 4) if n else None})
fill_pdf = pd.DataFrame(fill_records)
save_pdf(fill_pdf, "fill_rates")

fill_summary = (fill_pdf.groupby(["SOURCE", "OUTPUT"])
    .agg(ROWS=("ROWS", "first"), N_FEATURES=("COLUMN", "count"),
         MEDIAN_PCT_NON_NULL=("PCT_NON_NULL", "median"), MEDIAN_PCT_NON_ZERO=("PCT_NON_ZERO", "median"),
         N_100PCT_NULL=("PCT_NON_NULL", lambda s: int((s == 0).sum())),
         N_NEVER_NON_ZERO=("PCT_NON_ZERO", lambda s: int((s == 0).sum())),
         N_LT_1PCT_NON_ZERO=("PCT_NON_ZERO", lambda s: int((s < 1).sum())))
    .reset_index())
print("=== Fill-rate summary per output (monthly tables: model snapshot rows) ===")
print(fill_summary.to_string(index=False))

for _, r in fill_summary.iterrows():
    status = "WARN" if (r.N_100PCT_NULL > 0 or r.N_NEVER_NON_ZERO > 0) else "PASS"
    score("1 Fill rate", f"{r.SOURCE}_{r.OUTPUT}", status,
          f"{r.N_100PCT_NULL} cols 100% null, {r.N_NEVER_NON_ZERO} cols never non-zero, "
          f"{r.N_LT_1PCT_NON_ZERO} cols <1% non-zero (of {r.N_FEATURES})")

print("\n=== Columns that are 100% null or never non-zero ===")
dead = fill_pdf[(fill_pdf.PCT_NON_NULL == 0) | (fill_pdf.PCT_NON_ZERO == 0)]
print(dead[["SOURCE", "OUTPUT", "COLUMN", "PCT_NON_NULL", "PCT_NON_ZERO"]].to_string(index=False) if len(dead) else "  none")

print("\n=== 15 lowest-filled (non-zero) columns per output ===")
for (s, o), g in fill_pdf.groupby(["SOURCE", "OUTPUT"]):
    print(f"\n--- {s}_{o}")
    print(g.sort_values(["PCT_NON_ZERO", "PCT_NON_NULL"]).head(15)[["COLUMN", "PCT_NON_NULL", "PCT_NON_ZERO"]].to_string(index=False))
print(f"\nFull table: {EDA_OUT}/fill_rates")

In [0]:
RATIO_RE = re.compile(r'^(PERC|PCT|PERCENT|RATIO|SHARE|PROP|FRAC)_|_(PCT|PERC|PERCENT|RATIO|SHARE|PROP|FRAC)(_|$)')

ratio_records, nan_records = [], []
for k, df in outputs.items():                          # full written grain (all monthly rows)
    rc = [c for c in num_cols(df) if RATIO_RE.search(c) and not c.startswith("LOGRATIO")]
    dbl = [c for c, t in df.dtypes if t in ("double", "float") and c not in KEY_COLS]
    for ch in chunks(rc, CHUNK):
        exprs = []
        for i, c in enumerate(ch):
            x, isn = F.col(c).cast("double"), F.isnan(F.col(c).cast("double"))
            exprs += [F.min(F.when(~isn, x)).alias(f"mn{i}"), F.max(F.when(~isn, x)).alias(f"mx{i}"),
                      F.sum(F.when(~isn & (x < 0), 1).otherwise(0)).alias(f"lt{i}"),
                      F.sum(F.when(~isn & (x > 1), 1).otherwise(0)).alias(f"gt{i}"),
                      F.count(F.col(c)).alias(f"nn{i}")]
        r = df.agg(*exprs).first()
        for i, c in enumerate(ch):
            ratio_records.append({"SOURCE": k[0], "OUTPUT": k[1], "COLUMN": c,
                                  "NON_NULL": r[f"nn{i}"], "MIN": r[f"mn{i}"], "MAX": r[f"mx{i}"],
                                  "N_LT_0": r[f"lt{i}"], "N_GT_1": r[f"gt{i}"]})
    for ch in chunks(dbl, CHUNK):
        exprs = [F.sum(F.when(F.isnan(c) | (F.col(c) == float("inf")) | (F.col(c) == float("-inf")), 1)
                        .otherwise(0)).alias(f"bad{i}") for i, c in enumerate(ch)]
        r = df.agg(*exprs).first()
        for i, c in enumerate(ch):
            if r[f"bad{i}"]:
                nan_records.append({"SOURCE": k[0], "OUTPUT": k[1], "COLUMN": c, "N_NAN_OR_INF": r[f"bad{i}"]})

ratio_pdf, nan_pdf = pd.DataFrame(ratio_records), pd.DataFrame(nan_records)
save_pdf(ratio_pdf, "ratio_sanity"); save_pdf(nan_pdf, "nan_inf_columns")

print(f"=== Ratio / percentage features found: {len(ratio_pdf)} ===")
if len(ratio_pdf):
    ratio_pdf["STATUS"] = ratio_pdf.apply(lambda r: "FAIL" if (r.N_LT_0 or r.N_GT_1) else "PASS", axis=1)
    print(ratio_pdf.to_string(index=False))
    for (s, o), g in ratio_pdf.groupby(["SOURCE", "OUTPUT"]):
        bad = g[g.STATUS == "FAIL"]
        score("2 Ratio 0-1", f"{s}_{o}", "FAIL" if len(bad) else "PASS",
              f"{len(bad)} of {len(g)} ratio cols outside [0,1]" + (f": {list(bad.COLUMN)}" if len(bad) else ""))
else:
    print("  none")

print("\nNote: LOGRATIO_FREQ_* (ln ratio, unbounded by design) and SKEWNESS_* are not 0-1 features and are excluded.")
print(f"\n=== Double columns containing NaN / +-Inf: {len(nan_pdf)} ===")
print(nan_pdf.to_string(index=False) if len(nan_pdf) else "  none")
for k in outputs:
    g = nan_pdf[(nan_pdf.SOURCE == k[0]) & (nan_pdf.OUTPUT == k[1])] if len(nan_pdf) else nan_pdf
    score("2 NaN/Inf", lbl(k), "WARN" if len(g) else "PASS", f"{len(g)} double cols with NaN/Inf")

In [0]:
cov_records, per_cohort = [], []
for k, df in analysis.items():
    r = df.agg(F.count(F.lit(1)).alias("rows"),
               F.countDistinct("HCP_COHORT_ID").alias("ids"),
               F.sum(F.when(F.col("HCP_COHORT_ID").isNull() | F.col("PROVIDER_ID").isNull() |
                            F.col("COHORT").isNull(), 1).otherwise(0)).alias("null_keys"),
               F.sum(F.when(F.col("HCP_COHORT_ID") != F.concat_ws("_", "PROVIDER_ID", "COHORT"), 1)
                      .otherwise(0)).alias("bad_id")).first()
    pc = df.groupBy("COHORT").agg(F.count(F.lit(1)).alias("ROWS")).toPandas()
    present = set(pc.COHORT.dropna())
    miss_c  = [c for c in EXPECTED_COHORTS if c not in present]
    extra_c = sorted(present - set(EXPECTED_COHORTS))
    rec = {"SOURCE": k[0], "OUTPUT": k[1], "ROWS": r["rows"], "DISTINCT_HCP_COHORT_ID": r["ids"],
           "DUPLICATE_ROWS": r["rows"] - r["ids"], "NULL_KEYS": r["null_keys"],
           "ID_NOT_PROVIDER_COHORT": r["bad_id"], "N_COHORTS": len(present),
           "MISSING_COHORTS": ",".join(miss_c), "UNEXPECTED_COHORTS": ",".join(extra_c)}
    # monthly tables: grain + month-window check on the full written table
    if GRAIN[k[1]] == "hcp_month":
        m = (outputs[k].join(F.broadcast(cohort_sdf), "COHORT", "left")
             .groupBy("PROVIDER_ID", "COHORT")
             .agg(F.count(F.lit(1)).alias("n"), F.countDistinct("SVC_DT_M").alias("nm"),
                  F.first("N_MONTHS").alias("exp_m"),
                  F.sum(F.when((F.col("SVC_DT_M") < F.col("LB_START")) |
                               (F.col("SVC_DT_M") > F.col("LB_END")), 1).otherwise(0)).alias("out"))
             .agg(F.count(F.lit(1)).alias("hcp_cohorts"),
                  F.sum(F.col("n") - F.col("nm")).alias("dup_month_rows"),
                  F.sum(F.when(F.col("nm") != F.col("exp_m"), 1).otherwise(0)).alias("wrong_n_months"),
                  F.sum("out").alias("out_of_window")).first())
        rec.update({"MONTHLY_HCP_COHORTS": m["hcp_cohorts"], "MONTHLY_DUP_ROWS": m["dup_month_rows"],
                    "HCP_COHORTS_NOT_12_MONTHS": m["wrong_n_months"], "MONTH_ROWS_OUT_OF_WINDOW": m["out_of_window"],
                    "HCP_COHORTS_WITHOUT_SNAPSHOT_ROW": m["hcp_cohorts"] - r["ids"]})
    cov_records.append(rec)
    per_cohort.append(pc.set_index("COHORT")["ROWS"].rename(lbl(k)))

cov_pdf = pd.DataFrame(cov_records)
save_pdf(cov_pdf, "cohort_coverage")
print("=== Cohort coverage per output ===")
print(cov_pdf.T.to_string())

for _, r in cov_pdf.iterrows():
    issues = []
    if r.DUPLICATE_ROWS:          issues.append(f"{r.DUPLICATE_ROWS:,} duplicate HCP_COHORT_ID rows")
    if r.NULL_KEYS:               issues.append(f"{r.NULL_KEYS:,} null keys")
    if r.ID_NOT_PROVIDER_COHORT:  issues.append(f"{r.ID_NOT_PROVIDER_COHORT:,} ids != PROVIDER_ID_COHORT")
    if r.MISSING_COHORTS:         issues.append(f"missing cohorts {r.MISSING_COHORTS}")
    if r.UNEXPECTED_COHORTS:      issues.append(f"unexpected cohorts {r.UNEXPECTED_COHORTS}")
    for c in ("MONTHLY_DUP_ROWS", "HCP_COHORTS_NOT_12_MONTHS", "MONTH_ROWS_OUT_OF_WINDOW"):
        if c in r and pd.notna(r[c]) and r[c]:
            issues.append(f"{c}={int(r[c]):,}")
    score("3 Cohort coverage", f"{r.SOURCE}_{r.OUTPUT}", "FAIL" if issues else "PASS",
          "; ".join(issues) if issues else f"{r.N_COHORTS} cohorts, unique HCP_COHORT_ID")

per_cohort_pdf = pd.concat(per_cohort, axis=1).reindex(EXPECTED_COHORTS).fillna(0).astype("int64")
print("\n=== Rows (HCP-cohorts) per cohort per output ===")
print(per_cohort_pdf.to_string())
print("\n=== Same, as % of the HCP universe ===")
print((100 * per_cohort_pdf / n_universe).round(1).to_string())
save_pdf(per_cohort_pdf.reset_index().rename(columns={"index": "COHORT"}), "rows_per_cohort")

In [0]:
uni_ids = uni.select("PROVIDER_ID")
uni_records, missing_frames, id_frames = [], [], []
for k, df in analysis.items():
    ids   = df.select("PROVIDER_ID").distinct()
    n_ids = ids.count()
    n_in  = ids.join(uni_ids, "PROVIDER_ID", "left_semi").count()
    uni_records.append({"SOURCE": k[0], "OUTPUT": k[1], "DISTINCT_HCPS": n_ids, "IN_UNIVERSE": n_in,
                        "OUTSIDE_UNIVERSE": n_ids - n_in, "MISSING_FROM_OUTPUT": n_universe - n_in,
                        "PCT_UNIVERSE_COVERED": round(100 * n_in / n_universe, 2)})
    missing_frames.append(uni_ids.join(ids, "PROVIDER_ID", "left_anti")
                                 .withColumn("SOURCE", F.lit(k[0])).withColumn("OUTPUT", F.lit(k[1])))
    id_frames.append(ids.withColumn("SOURCE", F.lit(k[0])))

# per source (union of that notebook's outputs) and across all four notebooks
all_ids = reduce(lambda a, b: a.unionByName(b), id_frames)
for src in SOURCES + ["ALL"]:
    ids = (all_ids if src == "ALL" else all_ids.filter(F.col("SOURCE") == src)).select("PROVIDER_ID").distinct()
    n_ids = ids.count(); n_in = ids.join(uni_ids, "PROVIDER_ID", "left_semi").count()
    uni_records.append({"SOURCE": src, "OUTPUT": "ANY_OUTPUT", "DISTINCT_HCPS": n_ids, "IN_UNIVERSE": n_in,
                        "OUTSIDE_UNIVERSE": n_ids - n_in, "MISSING_FROM_OUTPUT": n_universe - n_in,
                        "PCT_UNIVERSE_COVERED": round(100 * n_in / n_universe, 2)})
    if src == "ALL":
        missing_frames.append(uni_ids.join(ids, "PROVIDER_ID", "left_anti")
                                     .withColumn("SOURCE", F.lit("ALL")).withColumn("OUTPUT", F.lit("ANY_OUTPUT")))

uni_pdf = pd.DataFrame(uni_records)
save_pdf(uni_pdf, "universe_coverage")
print(f"=== Universe coverage (universe = {n_universe:,} HCPs) ===")
print(uni_pdf.to_string(index=False))
for _, r in uni_pdf.iterrows():
    status = "FAIL" if r.OUTSIDE_UNIVERSE else ("PASS" if r.MISSING_FROM_OUTPUT == 0 else "INFO")
    score("4 Universe", f"{r.SOURCE}_{r.OUTPUT}", status,
          f"{r.IN_UNIVERSE:,}/{n_universe:,} HCPs ({r.PCT_UNIVERSE_COVERED}%), {r.OUTSIDE_UNIVERSE:,} outside universe")

# ---- which HCPs are missing: full list to Delta + profile by universe criteria ----
MISSING_PATH = f"{EDA_OUT}/missing_hcps"
(reduce(lambda a, b: a.unionByName(b), missing_frames)
   .write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(MISSING_PATH))
missing_hcps = spark.read.format("delta").load(MISSING_PATH)

profile = (missing_hcps.join(uni, "PROVIDER_ID", "left")
           .groupBy("SOURCE", "OUTPUT")
           .agg(F.count(F.lit(1)).alias("MISSING_HCPS"), *[F.sum(c).alias(f"of_which_{c}") for c in flag_cols])
           .orderBy("SOURCE", "OUTPUT").toPandas())
print("\n=== Missing HCPs profiled by universe criteria flags (C1..C5) ===")
print(profile.to_string(index=False))
save_pdf(profile, "missing_hcps_profile")

never = missing_hcps.filter(F.col("SOURCE") == "ALL")
print(f"\nHCPs in the universe that appear in NONE of the outputs: {never.count():,}  (sample of 20 below)")
print(never.join(uni, "PROVIDER_ID", "left").limit(20).toPandas().to_string(index=False))
print(f"\nFull missing-HCP list (PROVIDER_ID, SOURCE, OUTPUT): {MISSING_PATH}")

In [0]:
ID_PATH = f"{EDA_OUT}/hcp_cohort_id_presence"
(reduce(lambda a, b: a.unionByName(b),
        [df.select("HCP_COHORT_ID").distinct().withColumn("SRC", F.lit(k[0])).withColumn("FT", F.lit(k[1]))
         for k, df in analysis.items()])
   .write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(ID_PATH))
id_long = spark.read.format("delta").load(ID_PATH)

# ---- across notebooks (RX / DX / PX / LAB; union of each notebook's outputs) ----
by_src = id_long.groupBy("HCP_COHORT_ID").pivot("SRC", SOURCES).agg(F.count(F.lit(1)))
pattern = F.concat_ws("+", *[F.when(F.col(s) > 0, F.lit(s.upper())) for s in SOURCES])
pattern_pdf = (by_src.withColumn("PRESENT_IN", pattern).groupBy("PRESENT_IN").count()
                     .orderBy(F.desc("count")).toPandas())
total_ids = int(pattern_pdf["count"].sum())
pattern_pdf["PCT"] = (100 * pattern_pdf["count"] / total_ids).round(2)
print(f"=== Distinct HCP_COHORT_IDs across all outputs: {total_ids:,} ===")
print("Which notebooks each HCP_COHORT_ID appears in:")
print(pattern_pdf.to_string(index=False))
save_pdf(pattern_pdf, "cross_notebook_patterns")

r = by_src.agg(*[F.sum(F.when((F.col(a) > 0) & (F.col(b) > 0), 1).otherwise(0)).alias(f"{a}_{b}")
                 for a in SOURCES for b in SOURCES]).first()
overlap = pd.DataFrame([[r[f"{a}_{b}"] for b in SOURCES] for a in SOURCES],
                       index=[s.upper() for s in SOURCES], columns=[s.upper() for s in SOURCES])
print("\nPairwise overlap of HCP_COHORT_IDs (diagonal = IDs produced by that notebook):")
print(overlap.to_string())
in_all = int(pattern_pdf.loc[pattern_pdf.PRESENT_IN == "RX+DX+PX+LAB", "count"].sum())
score("5 Cross-notebook", "RX/DX/PX/LAB", "PASS" if in_all == total_ids else "INFO",
      f"{in_all:,} of {total_ids:,} HCP_COHORT_IDs ({100*in_all/builtins.max(total_ids,1):.1f}%) in all four notebooks; "
      "differences are expected (outputs only hold HCP-cohorts with activity in that claim type)")

# ---- within each notebook: which feature types produce which IDs + subset rules ----
print("\n=== Within-notebook ID patterns and subset rules ===")
for src in SOURCES:
    fts = [ft for ft in FEATURE_TYPES[src] if (src, ft) in analysis]
    if not fts: continue
    w = id_long.filter(F.col("SRC") == src).groupBy("HCP_COHORT_ID").pivot("FT", fts).agg(F.count(F.lit(1)))
    p = (w.withColumn("PRESENT_IN", F.concat_ws("+", *[F.when(F.col(ft) > 0, F.lit(ft)) for ft in fts]))
          .groupBy("PRESENT_IN").count().orderBy(F.desc("count")).toPandas())
    print(f"\n--- {src.upper()}")
    print(p.to_string(index=False))
    rules = [("recency", "first_occurrence"),   # a 12M event implies a first occurrence
             ("recency", "frequency"),          # a 12M event implies a frequency spine row
             ("hospitalization", "frequency")]  # hosp spine is built on the same 12M claims
    for a, b in rules:
        if a in fts and b in fts:
            v = w.filter((F.col(a) > 0) & F.col(b).isNull()).count()
            print(f"  rule {a} subset-of {b}: {v:,} violations")
            score("5 Cross-notebook", f"{src}: {a} in {b}", "PASS" if v == 0 else "WARN", f"{v:,} IDs in {a} but not in {b}")

#### Which notebooks produce which HCP_COHORT_IDs (by design)

The four notebooks are **not** expected to produce identical ID sets. Each output only contains HCP-cohorts that have activity in its own claim type:

| Output | Row exists when | Source input |
| --- | --- | --- |
| `*_first_occurrence` | the HCP has >= 1 basket event on or before the cohort cutoff (full history); all-null rows dropped | `{src}_full_history` |
| `*_recency` | the HCP has >= 1 basket event inside the cohort's 12-month lookback | `{src}_12m` |
| `*_frequency` (snapshot) | the HCP has >= 1 claim anywhere in the 12M input file (dense HCP x month spine), so it is the widest set per notebook | `{src}_12m` |
| `dx/px_hospitalization` (snapshot) | same spine as frequency; rows where TOTAL < HOSPITALIZED are removed | `{src}_12m` + `{src}_hosp` |

So `recency` should be a subset of `first_occurrence` and of `frequency`, and LAB (PCP-heavy lab feed) should cover the fewest HCPs. When joining into the base table, use the universe x cohort skeleton (SKILL.md Sec 8.2) and LEFT JOIN each output; missing IDs then become nulls, not dropped rows.

In [0]:
COUNT_RE = re.compile(r'^(FREQ_|NUM_OF_|TOTAL_|UNIQUE_)')
DAY_RE   = re.compile(r'^(MAX_EVENT_TIME_.*_FIRST_OCCURRENCE|MIN_EVENT_TIME_.*_TIME_MIN)$')

# ---- 6a. count / frequency features: >p99.9 outliers, implausible maxima, negatives ----
range_records = []
for k, df in analysis.items():
    cc = [c for c in num_cols(df) if COUNT_RE.match(c)]
    for ch in chunks(cc, CHUNK):
        r1 = df.agg(*[e for i, c in enumerate(ch) for e in (
                 F.percentile_approx(c, 0.999, 1000).alias(f"p{i}"), F.max(c).alias(f"mx{i}"),
                 F.sum(F.when(F.col(c) < 0, 1).otherwise(0)).alias(f"neg{i}"))]).first()
        p999 = {c: (r1[f"p{i}"] if r1[f"p{i}"] is not None else 0) for i, c in enumerate(ch)}
        r2 = df.agg(*[F.sum(F.when(F.col(c) > F.lit(p999[c]), 1).otherwise(0)).alias(f"gt{i}")
                      for i, c in enumerate(ch)]).first()
        for i, c in enumerate(ch):
            mx = r1[f"mx{i}"]
            range_records.append({"SOURCE": k[0], "OUTPUT": k[1], "COLUMN": c, "P99_9": p999[c], "MAX": mx,
                                  "MAX_OVER_P99_9": round(mx / builtins.max(p999[c], 1), 1) if mx is not None else None,
                                  "N_ABOVE_P99_9": r2[f"gt{i}"], "N_NEGATIVE": r1[f"neg{i}"],
                                  "IMPLAUSIBLE": bool(mx is not None and mx > OUTLIER_FACTOR * builtins.max(p999[c], 1))})
range_pdf = pd.DataFrame(range_records)
save_pdf(range_pdf, "count_feature_ranges")

print(f"=== 6a. Count / frequency features: {len(range_pdf)} columns checked ===")
rs = (range_pdf.groupby(["SOURCE", "OUTPUT"])
      .agg(N_COLS=("COLUMN", "count"), N_IMPLAUSIBLE=("IMPLAUSIBLE", "sum"),
           N_WITH_NEGATIVES=("N_NEGATIVE", lambda s: int((s > 0).sum())),
           ROWS_ABOVE_P99_9_TOTAL=("N_ABOVE_P99_9", "sum"), WORST_MAX_OVER_P99_9=("MAX_OVER_P99_9", "max"))
      .reset_index())
print(rs.to_string(index=False))
for _, r in rs.iterrows():
    status = "FAIL" if r.N_WITH_NEGATIVES else ("WARN" if r.N_IMPLAUSIBLE else "PASS")
    score("6 Range (counts)", f"{r.SOURCE}_{r.OUTPUT}", status,
          f"{int(r.N_IMPLAUSIBLE)} cols with max > {OUTLIER_FACTOR}x p99.9, {r.N_WITH_NEGATIVES} cols with negatives")
print(f"\nTop 30 most extreme columns (max / p99.9):")
print(range_pdf.sort_values("MAX_OVER_P99_9", ascending=False).head(30).to_string(index=False))

# ---- 6b. day features (EVENT_TIME, days before cutoff) inside the cohort window ----
#   first occurrence: 0 <= days <= datediff(last_day(lookback_end), DATA_START_DATE)
#   recency time_min: 0 <= days <= datediff(last_day(lookback_end), lookback_start)
day_records = []
ub_fo  = F.datediff(F.last_day("LB_END"), F.to_date(F.lit(DATA_START_DATE)))
ub_rec = F.datediff(F.last_day("LB_END"), F.col("LB_START"))
for k, df in analysis.items():
    dc = [c for c in num_cols(df) if DAY_RE.match(c)]
    if not dc: continue
    dj = df.join(F.broadcast(cohort_sdf), "COHORT", "left")
    for ch in chunks(dc, CHUNK):
        exprs = []
        for i, c in enumerate(ch):
            ub = ub_fo if c.startswith("MAX_EVENT_TIME_") else ub_rec
            exprs += [F.min(c).alias(f"mn{i}"), F.max(c).alias(f"mx{i}"),
                      F.sum(F.when(F.col(c) < 0, 1).otherwise(0)).alias(f"neg{i}"),
                      F.sum(F.when(F.col(c) > ub, 1).otherwise(0)).alias(f"ub{i}")]
        r = dj.agg(*exprs).first()
        for i, c in enumerate(ch):
            day_records.append({"SOURCE": k[0], "OUTPUT": k[1], "COLUMN": c, "MIN_DAYS": r[f"mn{i}"],
                                "MAX_DAYS": r[f"mx{i}"], "N_NEGATIVE": r[f"neg{i}"], "N_BEYOND_WINDOW": r[f"ub{i}"]})
day_pdf = pd.DataFrame(day_records)
save_pdf(day_pdf, "day_feature_ranges")
print(f"\n=== 6b. Day features: {len(day_pdf)} columns checked ===")
ds = (day_pdf.groupby(["SOURCE", "OUTPUT"])
      .agg(N_COLS=("COLUMN", "count"), MIN_DAYS=("MIN_DAYS", "min"), MAX_DAYS=("MAX_DAYS", "max"),
           COLS_WITH_NEGATIVE=("N_NEGATIVE", lambda s: int((s > 0).sum())),
           COLS_BEYOND_WINDOW=("N_BEYOND_WINDOW", lambda s: int((s > 0).sum())))
      .reset_index())
print(ds.to_string(index=False))
bad_days = day_pdf[(day_pdf.N_NEGATIVE > 0) | (day_pdf.N_BEYOND_WINDOW > 0)]
if len(bad_days): print(bad_days.to_string(index=False))
for _, r in ds.iterrows():
    bad = r.COLS_WITH_NEGATIVE + r.COLS_BEYOND_WINDOW
    score("6 Range (days)", f"{r.SOURCE}_{r.OUTPUT}", "FAIL" if bad else "PASS",
          f"days {r.MIN_DAYS}..{r.MAX_DAYS}; {r.COLS_WITH_NEGATIVE} cols negative, {r.COLS_BEYOND_WINDOW} cols beyond cohort window")

# ---- 6c. date columns inside the cohort range ----
max_end = builtins.max(cohort_pd.LB_END)
date_records = []
for k, df in outputs.items():
    dj = df.join(F.broadcast(cohort_sdf), "COHORT", "left")
    for c in [c for c, t in df.dtypes if t == "date"]:
        if c == "LOOKBACK_END_DATE":
            bad, rule = (F.col(c) != F.col("LB_END")), "== cohort lookback_end_date"
        elif c == "SVC_DT_M":
            bad, rule = ((F.col(c) < F.col("LB_START")) | (F.col(c) > F.col("LB_END"))), "within [lookback_start, lookback_end]"
        else:
            bad, rule = ((F.col(c) < F.to_date(F.lit(DATA_START_DATE))) | (F.col(c) > F.lit(max_end))), "within [data start, last cohort end]"
        r = dj.agg(F.min(c).alias("mn"), F.max(c).alias("mx"),
                   F.sum(F.when(bad, 1).otherwise(0)).alias("bad"),
                   F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias("nul"),
                   F.sum(F.when(F.col("LB_END").isNull(), 1).otherwise(0)).alias("nocoh")).first()
        date_records.append({"SOURCE": k[0], "OUTPUT": k[1], "COLUMN": c, "RULE": rule, "MIN": r["mn"], "MAX": r["mx"],
                             "N_VIOLATIONS": r["bad"], "N_NULL": r["nul"], "N_UNKNOWN_COHORT": r["nocoh"]})
date_pdf = pd.DataFrame(date_records)
save_pdf(date_pdf, "date_column_ranges")
print(f"\n=== 6c. Date columns vs cohort range ({cohort_pd.LB_START.min()} .. {max_end}) ===")
print(date_pdf.to_string(index=False))
for _, r in date_pdf.iterrows():
    bad = r.N_VIOLATIONS + r.N_NULL + r.N_UNKNOWN_COHORT
    score("6 Range (dates)", f"{r.SOURCE}_{r.OUTPUT}.{r.COLUMN}", "FAIL" if bad else "PASS",
          f"{r.MIN}..{r.MAX}; {r.N_VIOLATIONS:,} violations, {r.N_NULL:,} nulls ({r.RULE})")

In [0]:
def expected_columns(src, ft):
    ev = sorted({eda_sanitize(e) for e in event_groups(src)})
    if ft == "first_occurrence":                                    # Sec 3 + 8.1
        return ["HCP_COHORT_ID", "PROVIDER_ID", "COHORT"] + \
               [eda_sanitize(f"MAX_EVENT_TIME_{e}_FIRST_OCCURRENCE") for e in ev]
    if ft == "recency":                                             # Sec 4 (unsuffixed totals removed in 3.1)
        s = RECENCY_SUFFIX[src]
        return (["HCP_COHORT_ID", "PROVIDER_ID", "COHORT"]
                + [eda_sanitize(f"MIN_EVENT_TIME_{e}_TIME_MIN") for e in ev]
                + [f"TOTAL_{s}", f"UNIQUE_{s}"]
                + [f"{m}_EVENTS_TIME_{n}_DAYS_{s}" for m in ("TOTAL", "UNIQUE") for n in RECENCY_WINDOWS]
                + [f"SKEWNESS_TIME_{n}_DAYS_{s}" for n in RECENCY_WINDOWS])
    if ft == "frequency":                                           # Sec 5 A-D (+3.1 notes)
        return (["PROVIDER_ID", "COHORT", "SVC_DT_M"]
                + [f"FREQ_{e}_IN_LAST_{n}_MONTH" for e in ev for n in LAG_WINDOWS]
                + [f"LOGRATIO_FREQ_{e}_IN_LAST_{n}_MONTH" for e in ev for n in LOGRATIO_WINDOWS]
                + [f"{PATIENT_PREFIX[src]}_{e}_IN_LAST_{n}_DAYS" for e in ev for n in PATIENT_WINDOWS])
    if ft == "hospitalization":                                     # Sec 6 A-D
        ev1 = HOSP_EVENT[src]
        return (["PROVIDER_ID", "COHORT", "SVC_DT_M"]
                + [f"TOTAL_HOSPITALIZED_CLAIMS_{ev1}_IN_LAST_{n}_MONTH" for n in HOSP_LAG_WINDOWS]
                + [f"PERC_HOSPITALIZED_HCP_CLAIMS_FOR_{ev1}_IN_LAST_{n}_MONTH" for n in HOSP_LAG_WINDOWS]
                + [f"{HOSP_PAT_PREFIX[src]}_IN_LAST_{n}_DAYS" for n in HOSP_PAT_WINDOWS])

schema_records, drift_detail = [], []
for src, fts in FEATURE_TYPES.items():
    for ft in fts:
        exp = expected_columns(src, ft)
        if (src, ft) not in outputs:
            schema_records.append({"SOURCE": src, "OUTPUT": ft, "EXPECTED": len(exp), "ACTUAL": 0,
                                   "MISSING": len(exp), "EXTRA": 0, "STATUS": "FAIL (output missing)"})
            score("7 Schema drift", f"{src}_{ft}", "FAIL", "output missing"); continue
        act = outputs[(src, ft)].columns
        miss  = [c for c in exp if c not in set(act)]
        extra = [c for c in act if c not in set(exp)]
        dups  = len(act) - len(set(act))
        drift_detail += [{"SOURCE": src, "OUTPUT": ft, "KIND": "MISSING", "COLUMN": c} for c in miss]
        drift_detail += [{"SOURCE": src, "OUTPUT": ft, "KIND": "EXTRA", "COLUMN": c} for c in extra]
        status = "PASS" if not (miss or extra or dups) else ("FAIL" if miss else "WARN")
        schema_records.append({"SOURCE": src, "OUTPUT": ft, "EXPECTED": len(exp), "ACTUAL": len(act),
                               "MISSING": len(miss), "EXTRA": len(extra), "DUP_COLS": dups, "STATUS": status})
        score("7 Schema drift", f"{src}_{ft}", status, f"{len(miss)} missing, {len(extra)} extra vs SKILL.md")

schema_pdf, drift_pdf = pd.DataFrame(schema_records), pd.DataFrame(drift_detail)
save_pdf(schema_pdf, "schema_drift_summary"); save_pdf(drift_pdf, "schema_drift_columns")
print("=== Schema drift vs SKILL.md ===")
print(schema_pdf.to_string(index=False))

# Group the drifted columns by pattern so the reason is readable
def drift_family(c):
    c2 = re.sub(r'_IN_LAST_\d+_(MONTH|DAYS)$', r'_IN_LAST_{N}_\1', c)
    for p in ("MAX_EVENT_TIME_", "MIN_EVENT_TIME_", "LOGRATIO_FREQ_", "CHANGE_IN_FREQ_", "FREQ_",
              "NUM_OF_HOSPITALIZED_", "NUM_OF_PATIENTS_", "TOTAL_", "UNIQUE_", "SKEWNESS_", "PERC_"):
        if c.startswith(p):
            win = re.search(r'_IN_LAST_(\d+)_(MONTH|DAYS)$', c)
            return p + "*" + (f"_IN_LAST_{win.group(1)}_{win.group(2)}" if win else "")
    return c
if len(drift_pdf):
    drift_pdf["FAMILY"] = drift_pdf.COLUMN.map(drift_family)
    fam = drift_pdf.groupby(["SOURCE", "OUTPUT", "KIND", "FAMILY"]).agg(N=("COLUMN", "count"), EXAMPLE=("COLUMN", "first")).reset_index()
    print("\n=== Missing / extra columns grouped by family ===")
    print(fam.to_string(index=False))

# NBA 3.1 removal-note compliance (SKILL.md Sec 3, 4, 5A, 5C notes)
print("\n=== NBA 3.1 removal-note compliance ===")
cens = globals().get("CENSORED_EVENTS")
for (src, ft), df in outputs.items():
    cols = df.columns
    if ft == "recency":
        uns = [c for c in cols if c.startswith(("TOTAL_", "UNIQUE_")) and not c.endswith(("_RX", "_DX", "_PX", "_LAB_DATA"))]
        print(f"  {src}_{ft}: unsuffixed recency totals still present: {len(uns)}")
    if ft == "frequency":
        chg  = [c for c in cols if c.startswith("CHANGE_IN_FREQ_")]
        pw   = sorted({int(m.group(1)) for c in cols if c.startswith("NUM_OF_PATIENTS_")
                       for m in [re.search(r'_IN_LAST_(\d+)_DAYS$', c)] if m})
        lw   = sorted({int(m.group(1)) for c in cols if c.startswith("LOGRATIO_")
                       for m in [re.search(r'_IN_LAST_(\d+)_MONTH$', c)] if m})
        print(f"  {src}_{ft}: CHANGE_IN_FREQ_* present: {len(chg)} | LOGRATIO windows {lw} (skill {LOGRATIO_WINDOWS}) | "
              f"NUM_OF_PATIENTS windows {pw} (skill 3.1 note {PATIENT_WINDOWS})")
    if ft == "first_occurrence" and src == "rx":
        if cens:
            still = [e for e in cens if eda_sanitize(f"MAX_EVENT_TIME_{e}_FIRST_OCCURRENCE") in cols]
            print(f"  rx_first_occurrence: {len(still)} of {len(cens)} CENSORED_EVENTS still present")
        else:
            print("  rx_first_occurrence: CENSORED_EVENTS not defined in the config -> censored-event drop (Sec 3 note) not checkable")

# dtype expectations
print("\n=== dtype check ===")
for (src, ft), df in outputs.items():
    dt = dict(df.dtypes)
    wrong = [c for c in dt if (DAY_RE.match(c) and dt[c] not in ("int", "bigint")) or
             (c.startswith("PERC_") and dt[c] not in ("double", "float")) or
             (c in ("PROVIDER_ID", "COHORT", "HCP_COHORT_ID") and dt[c] != "string")]
    print(f"  {src}_{ft}: {len(wrong)} columns with unexpected dtype" + (f" e.g. {wrong[:5]}" if wrong else ""))

In [0]:
score_pdf = pd.DataFrame(SCORECARD)
save_pdf(score_pdf, "scorecard")
order = {"FAIL": 0, "WARN": 1, "INFO": 2, "PASS": 3}
score_pdf = score_pdf.assign(_o=score_pdf.STATUS.map(order)).sort_values(["_o", "CHECK", "OUTPUT"]).drop(columns="_o")

print("=== STATUS COUNTS ===")
print(score_pdf.groupby(["CHECK", "STATUS"]).size().unstack(fill_value=0).to_string())
print("\n=== FULL SCORECARD ===")
print(score_pdf.to_string(index=False))
print(f"\nAll EDA tables written under {EDA_OUT}/: fill_rates, ratio_sanity, nan_inf_columns, cohort_coverage, "
      "rows_per_cohort, universe_coverage, missing_hcps, missing_hcps_profile, hcp_cohort_id_presence, "
      "cross_notebook_patterns, count_feature_ranges, day_feature_ranges, date_column_ranges, "
      "schema_drift_summary, schema_drift_columns, scorecard")
display(score_pdf)